### Context Aware Music Recommender System

In [49]:
import pandas as pd

columns = ['userid', 'timestamp', 'artid', 'artname', 'traid', 'traname']

df = pd.read_csv(
        'data/lastfm-dataset-1K/userid-timestamp-artid-artname-traid-traname.tsv', 
        sep='\t', 
        on_bad_lines='skip',
        names=columns, 
        header=None,
        nrows=100000 # full load is 19 millions rows, cap it to 100,000
    )
df


,userid,timestamp,artid,artname,traid,traname
0,user_000001,2009-05-04T23:08:57Z,f1b1cf71-bd35-4e99-8624-24a6e15f133a,Deep Dish,NaN,Fuck Me Im Famous (Pacha Ibiza)-09-28-2007
1,user_000001,2009-05-04T13:54:10Z,a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,坂本龍一,NaN,Composition 0919 (Live_2009_4_15)
2,user_000001,2009-05-04T13:52:04Z,a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,坂本龍一,NaN,Mc2 (Live_2009_4_15)
3,user_000001,2009-05-04T13:42:52Z,a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,坂本龍一,NaN,Hibari (Live_2009_4_15)
4,user_000001,2009-05-04T13:42:11Z,a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,坂本龍一,NaN,Mc1 (Live_2009_4_15)
...,...,...,...,...,...,...
99995,user_000004,2008-03-20T14:08:49Z,61cd6f2a-cdde-4f34-aacd-b992badcacba,New Young Pony Club,9c7da1c0-108a-456a-baa2-e5d29e349324,Get Lucky
99996,user_000004,2008-03-20T14:04:57Z,61cd6f2a-cdde-4f34-aacd-b992badcacba,New Young Pony Club,27534cc1-6140-4c0a-82e2-dfc1b32bcdea,Tight Fit
99997,user_000004,2008-03-20T13:59:30Z,61cd6f2a-cdde-4f34-aacd-b992badcacba,New Young Pony Club,c271d600-46db-4f65-9231-dd5d334b12c8,Get Dancey
99998,user_000004,2008-03-20T13:55:22Z,61cd6f2a-cdde-4f34-aacd-b992badcacba,New Young Pony Club,9c7732f4-40e2-40ab-b113-b3b73bab51b5,Jerk Me


In [50]:
df.shape

(100000, 6)

In [51]:
df.columns

Index(['userid', 'timestamp', 'artid', 'artname', 'traid', 'traname'], dtype='str')

### Feature Extraction

In [52]:
# extract datetime
df["datetime"] = pd.to_datetime(df["timestamp"], unit="s")
df.head(3)

,userid,timestamp,artid,artname,traid,traname,datetime
0,user_000001,2009-05-04T23:08:57Z,f1b1cf71-bd35-4e99-8624-24a6e15f133a,Deep Dish,NaN,Fuck Me Im Famous (Pacha Ibiza)-09-28-2007,2009-05-04 23:08:57+00:00
1,user_000001,2009-05-04T13:54:10Z,a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,坂本龍一,NaN,Composition 0919 (Live_2009_4_15),2009-05-04 13:54:10+00:00
2,user_000001,2009-05-04T13:52:04Z,a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,坂本龍一,NaN,Mc2 (Live_2009_4_15),2009-05-04 13:52:04+00:00


In [53]:
# get hour and time_of_day
def get_time_of_day(hour):
    if 5 <= hour < 12:
        return "morning"
    elif 12 <= hour < 17:
        return "afternoon"
    elif 17 <= hour < 21:
        return "evening"
    else:
        return "night"

df["hour"] = df["datetime"].dt.hour
df["time_of_day"] = df["hour"].apply(get_time_of_day)
df.head(3)

,userid,timestamp,artid,artname,traid,traname,datetime,hour,time_of_day
0,user_000001,2009-05-04T23:08:57Z,f1b1cf71-bd35-4e99-8624-24a6e15f133a,Deep Dish,NaN,Fuck Me Im Famous (Pacha Ibiza)-09-28-2007,2009-05-04 23:08:57+00:00,23,night
1,user_000001,2009-05-04T13:54:10Z,a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,坂本龍一,NaN,Composition 0919 (Live_2009_4_15),2009-05-04 13:54:10+00:00,13,afternoon
2,user_000001,2009-05-04T13:52:04Z,a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,坂本龍一,NaN,Mc2 (Live_2009_4_15),2009-05-04 13:52:04+00:00,13,afternoon


In [54]:
# extract weekday & weekend
df["day_of_week"] = df["datetime"].dt.weekday
df["day_type"] = df["day_of_week"].apply(lambda x: "weekend" if x >= 5 else "weekday")
df.head(3)

,userid,timestamp,artid,artname,traid,traname,datetime,hour,time_of_day,day_of_week,day_type
0,user_000001,2009-05-04T23:08:57Z,f1b1cf71-bd35-4e99-8624-24a6e15f133a,Deep Dish,NaN,Fuck Me Im Famous (Pacha Ibiza)-09-28-2007,2009-05-04 23:08:57+00:00,23,night,0,weekday
1,user_000001,2009-05-04T13:54:10Z,a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,坂本龍一,NaN,Composition 0919 (Live_2009_4_15),2009-05-04 13:54:10+00:00,13,afternoon,0,weekday
2,user_000001,2009-05-04T13:52:04Z,a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,坂本龍一,NaN,Mc2 (Live_2009_4_15),2009-05-04 13:52:04+00:00,13,afternoon,0,weekday


In [55]:
# check if weekend is extracted
df[df['day_type'] == 'weekend'].head()

,userid,timestamp,artid,artname,traid,traname,datetime,hour,time_of_day,day_of_week,day_type
14,user_000001,2009-05-03T15:48:25Z,ba2f4f3b-0293-4bc8-bb94-2f73b5207343,Underworld,dc394163-2b78-4b56-94e4-658597a29ef8,"Boy, Boy, Boy (Switch Remix)",2009-05-03 15:48:25+00:00,15,afternoon,6,weekend
15,user_000001,2009-05-03T15:37:56Z,ba2f4f3b-0293-4bc8-bb94-2f73b5207343,Underworld,340d9a0b-9a43-4098-b116-9f79811bd508,Crocodile (Innervisions Orchestra Mix),2009-05-03 15:37:56+00:00,15,afternoon,6,weekend
16,user_000001,2009-05-03T15:14:53Z,a16e47f5-aa54-47fe-87e4-bb8af91a9fdd,Ennio Morricone,0b04407b-f517-4e00-9e6a-494795efc73e,Ninna Nanna In Blu (Raw Deal Remix),2009-05-03 15:14:53+00:00,15,afternoon,6,weekend
17,user_000001,2009-05-03T15:10:18Z,463a94f1-2713-40b1-9c88-dcc9c0170cae,Minus 8,4e78efc4-e545-47af-9617-05ff816d86e2,Elysian Fields,2009-05-03 15:10:18+00:00,15,afternoon,6,weekend
18,user_000001,2009-05-03T15:04:31Z,ad0811ea-e213-451d-b22f-fa1a7f9e0226,Beanfield,fb51d2c4-cc69-4128-92f5-77ec38d66859,Planetary Deadlock,2009-05-03 15:04:31+00:00,15,afternoon,6,weekend


##### Listening sessions
A session groups songs played close in time.
```
If gap between songs > 30 minutes → new session
```

In [56]:
# extract listening session

df = df.sort_values(['userid', 'datetime'])
df["time_diff"] = df.groupby("userid")["datetime"].diff().dt.seconds # in seconds

SESSION_GAP = 30 * 60 # in seconds

df["new_session"] = (df["time_diff"] > SESSION_GAP) | (df["time_diff"].isna())

df['session_id'] = df.groupby("userid")["new_session"].cumsum()

df.head(100)

,userid,timestamp,artid,artname,traid,traname,datetime,hour,time_of_day,day_of_week,day_type,time_diff,new_session,session_id
16684,user_000001,2006-08-13T13:59:20Z,09a114d9-7723-4e14-b524-379697f6d2b5,Plaid & Bob Jaroc,c4633ab1-e715-477f-8685-afa5f2058e42,The Launching Of Big Face,2006-08-13 13:59:20+00:00,13,afternoon,6,weekend,NaN,True,1
16683,user_000001,2006-08-13T14:03:29Z,09a114d9-7723-4e14-b524-379697f6d2b5,Plaid & Bob Jaroc,bc2765af-208c-44c5-b3b0-cf597a646660,Zn Zero,2006-08-13 14:03:29+00:00,14,afternoon,6,weekend,249.0,False,1
16682,user_000001,2006-08-13T14:10:43Z,09a114d9-7723-4e14-b524-379697f6d2b5,Plaid & Bob Jaroc,aa9c5a80-5cbe-42aa-a966-eb3cfa37d832,The Return Of Super Barrio - End Credits,2006-08-13 14:10:43+00:00,14,afternoon,6,weekend,434.0,False,1
16681,user_000001,2006-08-13T14:17:40Z,67fb65b5-6589-47f0-9371-8a40eb268dfb,Tommy Guerrero,d9b1c1da-7e47-4f97-a135-77260f2f559d,Mission Flats,2006-08-13 14:17:40+00:00,14,afternoon,6,weekend,417.0,False,1
16680,user_000001,2006-08-13T14:19:06Z,1cfbc7d1-299c-46e6-ba4c-1facb84ba435,Artful Dodger,120bb01c-03e4-465f-94a0-dce5e9fac711,What You Gonna Do?,2006-08-13 14:19:06+00:00,14,afternoon,6,weekend,86.0,False,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16589,user_000001,2006-08-16T11:10:38Z,07729ca8-b15b-44d4-881b-6a325f61665a,Maysa,d1bdae20-9ba3-4c49-ae3c-99089cdd6d1a,Blue Horizon,2006-08-16 11:10:38+00:00,11,morning,2,weekday,62.0,False,3
16588,user_000001,2006-08-16T11:14:54Z,07729ca8-b15b-44d4-881b-6a325f61665a,Maysa,bbb904d2-d28b-4ecf-933a-6aa6720d71de,Osaka,2006-08-16 11:14:54+00:00,11,morning,2,weekday,256.0,False,3
16587,user_000001,2006-08-16T11:16:34Z,07729ca8-b15b-44d4-881b-6a325f61665a,Maysa,b37a264d-4f4d-4289-b2c4-8a9dc102171d,Everything,2006-08-16 11:16:34+00:00,11,morning,2,weekday,100.0,False,3
16586,user_000001,2006-08-16T13:43:02Z,07729ca8-b15b-44d4-881b-6a325f61665a,Maysa,4d201e6a-e823-4ed0-80a3-ebc32492c369,Tailor Made Love,2006-08-16 13:43:02+00:00,13,afternoon,2,weekday,8788.0,True,4


##### User taste
User taste means what artists or songs the user prefers.
```
play_count
```

In [57]:
# extract user prefer artist
user_artist_pref = (
    df.groupby(["userid", "artname"])
      .size()
      .reset_index(name="play_count")
)
user_artist_pref.sort_values("play_count", ascending=False) # top artist

,userid,artname,play_count
1738,user_000002,The Libertines,2411
762,user_000002,Babyshambles,1724
1235,user_000002,Kettcar,1282
1734,user_000002,The Kooks,978
1315,user_000002,Maxïmo Park,941
...,...,...,...
3829,user_000004,Xiu Xiu,1
3831,user_000004,Yael Naïm,1
3833,user_000004,Yelle,1
3834,user_000004,Yo La Tengo,1


#### Feature extraction summary
From the original columns you can derive:

From `timestamp`

- time_of_day
- weekday/weekend
- hour
- month
- season

From `timestamp gaps`

- session_id
- session_length
- session_position

From `listening history`

- user favorite artists
- user favorite songs
- artist popularity